# OOPSIE: Real-GPU Service Integration Test

This notebook performs the **first real-GPU end-to-end integration test** for the OOPSIE inference service on a Kaggle NVIDIA GPU (Tesla T4 or higher).

> **STRICT SCOPE & CONSTRAINTS**:
> - **No Retraining / No Merging**: Zero weights altered or merged.
> - **Imports Actual Code**: Imports repository modules directly from the service bundle (no duplicate implementation cells).
> - **In-Process Testing**: Uses FastAPI `TestClient` in-process; no public tunnels or open public ports.
> - **No Silent Healing**: Strict raw completion parsing with zero markdown fence stripping or syntax repair.

## Step 1: Install Dependencies

Installs the exact verified package versions from previous Kaggle runs (`env_packages.json`), plus FastAPI and Uvicorn for service testing.

In [ ]:
!pip install -q -U git+https://github.com/huggingface/transformers.git
!pip install -q -U trl>=0.12.0 peft>=0.14.0 bitsandbytes>=0.45.0 accelerate datasets
!pip install -q -U fastapi>=0.115.0 uvicorn httpx
print("Dependencies installed successfully.")

## Step 2: Unpack Service Code Bundle

Extracts `oopsie-service-code.zip` into `/kaggle/working/service_root` and adds it to `sys.path` so the notebook imports the actual repository inference engine and API.

In [ ]:
import os
import sys
import zipfile
from pathlib import Path

candidate_dirs = [p.parent.parent for p in Path("/kaggle/input").rglob("inference/config.py")]
if candidate_dirs:
    service_root = candidate_dirs[0]
    print(f"Found unpacked service code in Kaggle input: {service_root}")
else:
    zip_matches = list(Path("/kaggle/input").rglob("*.zip"))
    if zip_matches:
        bundle_path = zip_matches[0]
        print(f"Extracting service bundle from: {bundle_path}")
        service_root = Path("/kaggle/working/service_root")
        service_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(bundle_path, "r") as zf:
            zf.extractall(service_root)
    else:
        raise FileNotFoundError(f"Service code not found under /kaggle/input! Contents: {list(Path('/kaggle/input').glob('*'))}")

if str(service_root) not in sys.path:
    sys.path.insert(0, str(service_root))

# Verify imports from actual code
import inference.config as cfg
from inference.engine import BaseInferenceEngine, Gemma4PilotEngine, MockInferenceEngine, set_engine
from inference.parser import strict_parse_raw_completion, validate_oopsie_report
from inference.api.main import app, get_semaphore

print("Service code successfully unpacked and imported from:", service_root)

## Step 3: Hardware Preflight & Tesla T4 Compatibility Patch

In [ ]:
import torch

# Tesla T4 (CC 7.5) compatibility patch: avoid torch.addmm(out_dtype=torch.float32) on BFloat16
try:
    import trl.kernels.chunked_logprob
    trl.kernels.chunked_logprob._MM_OUT_DTYPE = False
    print("Applied Tesla T4 kernel compatibility workaround (_MM_OUT_DTYPE = False).")
except Exception as e:
    print(f"TRL kernel patch notice: {e}")

assert torch.cuda.is_available(), "CUDA GPU accelerator required!"
device_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"Primary GPU   : {device_name}")
print(f"Available VRAM: {vram_gb:.2f} GB")
assert vram_gb >= 14.0, f"Insufficient VRAM: {vram_gb:.2f} GB (minimum 14.0 GB required for 12B NF4)."

# HF_TOKEN check from Kaggle Secrets (do not print token)
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        os.environ["HF_TOKEN"] = hf_token
        print("Successfully loaded HF_TOKEN from Kaggle Secrets.")
    except Exception:
        raise RuntimeError("HF_TOKEN not found! Add via Kaggle: Add-ons -> Secrets -> HF_TOKEN.")
else:
    print("HF_TOKEN detected in environment.")

## Step 4: Controlled Concurrency & Cancellation Pre-Check (Fake Engine)

Runs a controlled fake-engine test before loading GPU weights to verify that:
1. Asynchronous health endpoints remain responsive during generation.
2. Admission bounds reject concurrent requests with `HTTP 429 CAPACITY_LIMIT`.
3. Client cancellation does not prematurely release GPU admission.

In [ ]:
import time
import threading
from starlette.testclient import TestClient

class ControlledSlowEngine(MockInferenceEngine):
    def __init__(self, delay_s: float = 0.3):
        super().__init__(ready=True)
        self.delay_s = delay_s
        self.started_event = threading.Event()
        self.completed_event = threading.Event()

    def generate(self, raw_error: str, context=None, formatter_id=cfg.FORMATTER_TRAINING_ALIGNED):
        self.started_event.set()
        time.sleep(self.delay_s)
        self.completed_event.set()
        return super().generate(raw_error, context, formatter_id)

slow_engine = ControlledSlowEngine(delay_s=0.3)
set_engine(slow_engine)

with TestClient(app) as tc:
    t1_res = {}
    def run_worker():
        t1_res["resp"] = tc.post("/v1/analyze", json={"error_text": "precheck test log"})

    th = threading.Thread(target=run_worker)
    th.start()
    assert slow_engine.started_event.wait(timeout=2.0), "Engine failed to start generation."

    # 1. Health endpoint responsiveness check
    t0 = time.perf_counter()
    h_resp = tc.get("/health")
    h_dur = time.perf_counter() - t0
    assert h_resp.status_code == 200, f"Health check failed: {h_resp.status_code}"
    assert h_dur < 0.15, f"Health check blocked by generation! Duration: {h_dur:.3f}s"
    print(f"Precheck 1 PASS: Health endpoint responsive during generation ({h_dur * 1000:.1f}ms).")

    # 2. Concurrency rejection check
    c_resp = tc.post("/v1/analyze", json={"error_text": "concurrent request"})
    assert c_resp.status_code == 429, f"Expected 429 CAPACITY_LIMIT, got {c_resp.status_code}"
    assert c_resp.json()["error"]["code"] == "CAPACITY_LIMIT"
    print("Precheck 2 PASS: Concurrent request rejected with HTTP 429 CAPACITY_LIMIT.")

    # 3. Cancellation / completion bounding check
    sem = get_semaphore()
    assert sem.acquire(blocking=False) is False, "Semaphore prematurely released while generation running!"
    th.join()
    assert t1_res["resp"].status_code == 200, "Primary request failed."
    assert sem.acquire(blocking=False) is True, "Semaphore not released after generation completion!"
    sem.release()
    print("Precheck 3 PASS: GPU capacity held until background thread completion.")

print("All concurrency pre-checks PASSED.")

## Step 5: Configure the Real Engine & Mount Points

Locates the pilot adapter weights under `/kaggle/input`, disables mock mode, and configures `Gemma4PilotEngine`.

In [ ]:
# Locate adapter directory containing adapter_model.safetensors
adapter_files = list(Path("/kaggle/input").rglob("adapter_model.safetensors"))
if not adapter_files:
    raise FileNotFoundError(
        "adapter_model.safetensors not found under /kaggle/input! Add the pilot adapter dataset to this notebook."
    )
adapter_path = adapter_files[0].parent
print(f"Located pilot adapter directory: {adapter_path}")

# Explicit configuration
os.environ["OOPSIE_USE_MOCK_ENGINE"] = "0"
os.environ["OOPSIE_MODEL_PATH"] = str(adapter_path)
os.environ["OOPSIE_BASE_MODEL_ID"] = "google/gemma-4-12b-it"
os.environ["OOPSIE_FORMATTER"] = cfg.FORMATTER_TRAINING_ALIGNED

assert not cfg.is_mock_engine_enabled(), "Mock mode must be explicitly disabled!"

real_engine = Gemma4PilotEngine(
    adapter_path=adapter_path,
    base_model_id="google/gemma-4-12b-it",
)
assert isinstance(real_engine, Gemma4PilotEngine), "Engine must be Gemma4PilotEngine!"
set_engine(real_engine)

print("Configured real Gemma4PilotEngine with training_aligned_v1 formatter.")

## Step 6: Load Real Model Weights & Probe Readiness

Loads the 12B base model in 4-bit NF4 with double quantization and attaches the LoRA adapter once.

In [ ]:
print("Loading base model and pilot adapter into GPU...")
t0_load = time.perf_counter()
real_engine.load()
load_duration = time.perf_counter() - t0_load

assert real_engine.is_ready() is True, "Model engine failed to report ready!"
allocated_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
reserved_gb = torch.cuda.memory_reserved(0) / (1024 ** 3)

print(f"Model loaded in {load_duration:.2f} seconds.")
print(f"GPU Memory Allocated: {allocated_gb:.2f} GB")
print(f"GPU Memory Reserved : {reserved_gb:.2f} GB")

# Verify API readiness endpoint inside TestClient context manager
with TestClient(app) as client:
    ready_resp = client.get("/ready")
    assert ready_resp.status_code == 200, f"Readiness probe returned {ready_resp.status_code}"
    ready_data = ready_resp.json()
    assert ready_data["status"] == "ready"
    assert ready_data["model_ready"] is True
    assert ready_data["model_version"] == cfg.MODEL_VERSION
    print("API Readiness Probe: READY (HTTP 200).")

## Step 7: Execute Real Request through POST /v1/analyze

Submits an ordinary synthetic error log through the API service. Captures complete outer envelopes, token counts, throughput, and parses output strictly.

In [ ]:
sample_request = {
    "error_text": (
        "[eslint] Warning: 'unusedVar' is defined but never used (no-unused-vars) in src/Header.js:3:7\n"
        "[build] 2 warnings found.\n"
        "[builder] Starting compiler step...\n"
        "[builder] Process finished with exit code 1."
    ),
    "context": {
        "language_or_framework": "React / Webpack",
        "environment": "ci_build"
    },
    "formatter": "training_aligned_v1"
}

print("Submitting real inference request to POST /v1/analyze...")
t0_req = time.perf_counter()
with TestClient(app) as client:
    resp = client.post("/v1/analyze", json=sample_request)
total_req_time = (time.perf_counter() - t0_req) * 1000.0

print(f"HTTP Status Code: {resp.status_code}")
body = resp.json()

request_success = (resp.status_code == 200 and body.get("status") == "success")
if request_success:
    print("\n--- OUTER SUCCESS ENVELOPE ---")
    print(f"Request ID        : {body['request_id']}")
    print(f"Model Version     : {body['model_version']}")
    print(f"Formatter Version : {body['formatter_version']}")
    print(f"Prompt Tokens     : {body['metadata']['prompt_tokens']}")
    print(f"Generated Tokens  : {body['metadata']['generated_tokens']}")
    print(f"Budget Hit (Cutoff): {body['metadata']['budget_hit']}")
    print(f"Measured Latency  : {body['metadata']['duration_ms']} ms")
    tok_per_sec = body['metadata']['generated_tokens'] / (body['metadata']['duration_ms'] / 1000.0)
    print(f"Throughput        : {tok_per_sec:.2f} tok/s")
    
    data = body["data"]
    print("\n--- VALIDATED OOPSIE REPORT ---")
    print(f"Summary           : {data.get('summary')}")
    print(f"Risk Level        : {data.get('risk_level')}")
    print(f"Error Category    : {data.get('error_category')}")
    print(f"Likely Causes     : {len(data.get('likely_causes', []))} item(s)")
    print(f"Repair Prompt     : {data.get('repair_prompt')[:140]}...")
else:
    print("\n--- SERVICE FAILURE ENVELOPE ---")
    print(f"Request ID        : {body.get('request_id')}")
    print(f"Error Code        : {body.get('error', {}).get('code')}")
    print(f"Error Message     : {body.get('error', {}).get('message')}")

## Step 8: Save Integration Report & Artifacts

Records the real GPU integration outcomes into `/kaggle/working/oopsie-gpu-integration/`.

In [ ]:
import json
from pathlib import Path
out_dir = Path("/kaggle/working/oopsie-gpu-integration")
out_dir.mkdir(parents=True, exist_ok=True)

integration_report = {
    "service_status": "integration_test_completed",
    "hardware": {
        "device": device_name,
        "vram_total_gb": round(vram_gb, 2),
        "vram_allocated_gb": round(allocated_gb, 2),
        "vram_reserved_gb": round(reserved_gb, 2),
    },
    "model_configuration": {
        "base_model_id": "google/gemma-4-12b-it",
        "adapter_path": str(adapter_path),
        "engine_type": type(real_engine).__name__,
        "mock_mode_disabled": not cfg.is_mock_engine_enabled(),
        "formatter": cfg.FORMATTER_TRAINING_ALIGNED,
        "max_new_tokens": cfg.MAX_NEW_TOKENS,
        "temperature": cfg.TEMPERATURE,
        "do_sample": cfg.DO_SAMPLE,
    },
    "lifecycle_checks": {
        "model_load_duration_s": round(load_duration, 2),
        "readiness_status_code": ready_resp.status_code,
        "readiness_body": ready_data,
        "concurrency_prechecks_passed": True,
    },
    "inference_request_outcome": {
        "http_status": resp.status_code,
        "envelope_status": body.get("status"),
        "request_id": body.get("request_id"),
        "metadata": body.get("metadata"),
        "report_validated": request_success,
        "error_details": body.get("error") if not request_success else None,
    },
    "limitations_notice": (
        "This single integration test confirms API and model execution on real GPU hardware. "
        "It does NOT establish a new benchmark, prove repair correctness, or justify public deployment."
    )
}

with open(out_dir / "gpu_service_integration_report.json", "w", encoding="utf-8") as f:
    json.dump(integration_report, f, indent=2)

# Diagnostic artifact containing raw completion for debugging (excluded from service logs)
with open(out_dir / "api_response_envelope.json", "w", encoding="utf-8") as f:
    json.dump(body, f, indent=2)

summary_md = f"""# OOPSIE Real-GPU Integration Test Summary

- **Date**: {time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())}
- **GPU**: {device_name} ({vram_gb:.2f} GB)
- **Base Model**: `google/gemma-4-12b-it`
- **Adapter**: `{adapter_path}`
- **Engine**: `{type(real_engine).__name__}` (Mock Mode: False)
- **Formatter**: `{cfg.FORMATTER_TRAINING_ALIGNED}`
- **Load Time**: {load_duration:.2f}s (Allocated: {allocated_gb:.2f} GB)
- **API Readiness Probe**: HTTP {ready_resp.status_code}
- **Inference Outcome**: HTTP {resp.status_code} ({body.get('status')})
- **Generated Tokens**: {body.get('metadata', {}).get('generated_tokens', 'N/A')}
- **Throughput**: {tok_per_sec:.2f} tok/s if request_success else 'N/A'

> This artifact validates operational service contracts on real GPU hardware without public deployment.
"""

with open(out_dir / "integration_summary.md", "w", encoding="utf-8") as f:
    f.write(summary_md)

print(f"\nArtifacts successfully written to {out_dir}:")
for item in out_dir.iterdir():
    print(f"  - {item.name} ({item.stat().st_size:,} bytes)")
print("\nReal-GPU integration test harness completed.")